# No-gaze analysis pipeline

This notebook reproduces the steps used for:

- reading the CSV files
- normalizing Student IDs
- merging assignment + pre + post data
- building questionnaire composites
- computing reliability (Cronbach's alpha)
- performing manipulation checks
- reshaping ad-level responses
- running the no-gaze hypothesis tests
- drawing figures from the analysis results

It is written for the files currently stored in `/mnt/data`.


In [2]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import statsmodels.formula.api as smf
import statsmodels.api as sm

from pathlib import Path

ASSIGN_PATH = Path("assignment_MAX45_beverage_can_repeat - Summary.csv")
PRE_PATH    = Path("Copy of Pre-Experiment Questionnaire (事前質問) (Responses) - Form Responses 1 (1).csv")
POST_PATH   = Path("Copy of Post-Experiment Questionnaire (事後質問) (Responses) - Form Responses 1 (1).csv")

OUTPUT_DIR = Path("analysis_outputs_no_gaze")
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 200)


## 1. Helper functions

In [8]:
def read_csv_fallback(path):
    for enc in ["utf-8", "utf-8-sig", "cp932", "shift_jis", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    raise ValueError(f"Could not read CSV: {path}")

def normalize_text(s):
    if pd.isna(s):
        return np.nan
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.strip()
    return s

def normalize_id(s):
    if pd.isna(s):
        return np.nan
    s = normalize_text(s)
    s = s.upper()
    s = re.sub(r"\s+", "", s)
    return s

def normalize_group(s):
    if pd.isna(s):
        return np.nan

    s = normalize_text(s).lower()

    # smartwatch first
    if "smartwatch" in s or "smart watch" in s or "watch" in s:
        return "SMARTWATCH"

    # control
    if "control" in s or "none" in s:
        return "CONTROL"

    # AR more strictly
    if s == "ar" or " ar " in f" {s} " or "augmented reality" in s:
        return "AR"

    return s.upper()

def find_cols(df, patterns, exclude=None):
    exclude = exclude or []
    cols = []
    for c in df.columns:
        c0 = str(c)
        if any(p.lower() in c0.lower() for p in patterns) and not any(e.lower() in c0.lower() for e in exclude):
            cols.append(c)
    return cols

def cronbach_alpha(df_items):
    x = df_items.dropna()
    if x.shape[0] < 2 or x.shape[1] < 2:
        return np.nan
    item_vars = x.var(axis=0, ddof=1)
    total_var = x.sum(axis=1).var(ddof=1)
    k = x.shape[1]
    if total_var == 0:
        return np.nan
    return (k / (k - 1.0)) * (1 - item_vars.sum() / total_var)

def reverse_code(series, min_val, max_val):
    return max_val + min_val - pd.to_numeric(series, errors="coerce")

def safe_mean(df, cols):
    return df[cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

def to_numeric_df(df, cols):
    out = df[cols].copy()
    for c in cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def parse_yes_no_not_sure(x):
    if pd.isna(x):
        return np.nan
    s = normalize_text(x).lower()
    if "yes" in s or "はい" in s:
        return 1
    if "no" in s or "いいえ" in s:
        return 0
    if "not sure" in s or "わから" in s:
        return np.nan
    return np.nan

def parse_manipulation_group(x):
    if pd.isna(x):
        return np.nan
    s = normalize_text(x).lower()
    if "not sure" in s or "わから" in s:
        return "NOT_SURE"
    if "ar" in s:
        return "AR"
    if "smart" in s or "watch" in s:
        return "SMARTWATCH"
    if "in-store only" in s or "no wearable" in s or "none" in s:
        return "CONTROL"
    return np.nan

def winner_counts_from_tlx(df, pair_cols):
    dims = {
        "mental": ["mental demand", "知的"],
        "physical": ["physical demand", "身体的要求"],
        "temporal": ["temporal demand", "タイム"],
        "performance": ["performance", "作業成績"],
        "effort": ["effort", "努力"],
        "frustration": ["frustration", "フラスト"],
    }
    def which_dim(value):
        s = normalize_text(value).lower()
        for dim, keys in dims.items():
            if any(k.lower() in s for k in keys):
                return dim
        return np.nan
    out = pd.DataFrame(index=df.index)
    for dim in dims:
        out[f"weight_{dim}"] = 0
    for c in pair_cols:
        chosen = df[c].apply(which_dim)
        for dim in dims:
            out.loc[chosen == dim, f"weight_{dim}"] += 1
    return out

def weighted_tlx(df, rating_cols, pair_cols):
    ratings = pd.DataFrame(index=df.index)
    for dim, col in rating_cols.items():
        ratings[dim] = pd.to_numeric(df[col], errors="coerce")
    weights = winner_counts_from_tlx(df, pair_cols)
    num = (
        ratings["mental"] * weights["weight_mental"] +
        ratings["physical"] * weights["weight_physical"] +
        ratings["temporal"] * weights["weight_temporal"] +
        ratings["performance"] * weights["weight_performance"] +
        ratings["effort"] * weights["weight_effort"] +
        ratings["frustration"] * weights["weight_frustration"]
    )
    den = weights.sum(axis=1).replace(0, np.nan)
    return num / den

def robust_ols(formula, data):
    return smf.ols(formula, data=data).fit(cov_type="HC3")

def clustered_ols(formula, data, cluster_col):
    return smf.ols(formula, data=data).fit(
        cov_type="cluster",
        cov_kwds={"groups": data[cluster_col]}
    )

def robust_logit(formula, data):
    model = smf.logit(formula, data=data).fit(disp=False)
    return model._get_robustcov_results(cov_type="HC3")

def omnibus_p_for_factor(model, factor_prefix):
    names = [n for n in model.params.index if factor_prefix in n]
    if not names:
        return np.nan
    R = np.zeros((len(names), len(model.params)))
    for i, name in enumerate(names):
        R[i, list(model.params.index).index(name)] = 1
    w = model.wald_test(R)
    return float(w.pvalue)

def product_root(ad_str):
    if pd.isna(ad_str):
        return np.nan
    s = normalize_text(ad_str)
    s = re.sub(r"_(NONE|BOGO|50OFF|50_OFF)$", "", s, flags=re.IGNORECASE)
    return s

def parse_discount_from_ad(ad_str):
    if pd.isna(ad_str):
        return np.nan
    s = normalize_text(ad_str).upper()
    if "BOGO" in s:
        return "BOGO"
    if "50OFF" in s or "50_OFF" in s or "50%" in s:
        return "50OFF"
    if "NONE" in s:
        return "NONE"
    return np.nan

BEVERAGES = {
    "STARBUCKS_LATTE_CAN",
    "STARBUCKS_BLACK_CAN",
    "SPRITE_CAN",
    "COCA_COLA_ZERO_CAN",
    "FANTA_GRAPE_CAN",
    "FANTA_ORANGE_CAN",
    "MITSUYA_BOTTLE",
    "OICHA_BOTTLE",
    "FANTA_GRAPE_BOTTLE",
    "COLA_BOTTLE",
    "FANTA_ORANGE_BOTTLE",
    "CALPIS_BOTTLE",
}

CHOCOLATES = {
    "ALFORT_BLUE",
    "ALFORT_WHITE",
    "ALFORT_BROWN",
    "CRUNKY_STRAWBERRY",
    "CRUNKY_CHOCO",
    "COOKIE_CHOCOCHIP",
    "COOKIE_WHITE",
    "COOKIE_ALMOND",
    "COOKIE_MOONLIGHT",
    "COOKIE_BLACKLIGHT",
    "GHANA_RED",
    "GHANA_BLACK",
    "GHANA_WHITE",
    "GHANA_ORANGE",
    "COALA_MARCH_ORIGINAL",
    "COALA_MARCH_STRAWBERRY",
    "COALA_MARCH_BARLEY",
    "POCKY_ORIGINAL",
    "POCKY_ALMOND",
    "POCKY_MATCHA",
    "POCKY_STRAWBERRY",
    "POCKY_WHITE",
    "CHOCO_BISCUIT",
    "KITKAT_PURPLE",
    "KITKAT_AUTUMN",
    "KITKAT_RED",
    "KITKAT_MATCHA",
    "KITKAT_LATTE",
    "KITKAT_BLACK",
}

SNACKS = {
    "NOODLE_CUP_ORANGE",
    "NOODLE_CUP_RED",
    "NOODLE_CUP_SEAFOOD",
    "NOODLE_CUP_CLASSIC",
    "NOODLE_CUP_TOMATO",
    "NOODLE_CUP_BLACK",
    "RED_DORITOS_BAG",
    "BLUE_BAG",
    "PINK_BAG",
    "YELLOW_BAG",
    "KARAMUCHO_BAG",
    "WASABI_BAG",
    "PRETZ_ORANGE",
    "PRETZ_PURPLE",
    "PRETZ_TOMATO",
}

def product_category_from_root(root):
    if pd.isna(root):
        return np.nan
    r = normalize_text(root).upper()
    if r in BEVERAGES:
        return "BEVERAGE"
    if r in CHOCOLATES:
        return "CHOCOLATE"
    if r in SNACKS:
        return "SNACK"
    return np.nan

def normalize_category_response(x):
    if pd.isna(x):
        return np.nan
    s = normalize_text(x).lower()
    if "beverage" in s:
        return "BEVERAGE"
    if "chocolate" in s:
        return "CHOCOLATE"
    if "chips" in s or "snack" in s or "cup noodles" in s:
        return "SNACK"
    if "not sure" in s:
        return np.nan
    return np.nan

def normalize_discount_response(x):
    if pd.isna(x):
        return np.nan
    s = normalize_text(x).upper()
    if "BOGO" in s:
        return "BOGO"
    if "50" in s:
        return "50OFF"
    if "NO DISCOUNT" in s or "NONE" in s:
        return "NONE"
    if "NOT SURE" in s:
        return np.nan
    return np.nan

def normalize_brand_response(x):
    if pd.isna(x):
        return np.nan
    s = normalize_text(x).upper()
    s = re.sub(r"[^A-Z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def normalize_true_root(root):
    if pd.isna(root):
        return np.nan
    s = normalize_text(root).upper()
    s = re.sub(r"[^A-Z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def brand_match(resp_brand, true_root):
    rb = normalize_brand_response(resp_brand)
    tr = normalize_true_root(true_root)
    if pd.isna(rb) or pd.isna(tr):
        return np.nan
    if rb == tr:
        return 1
    if rb in tr or tr in rb:
        return 1
    rb_tokens = set(rb.split("_"))
    tr_tokens = set(tr.split("_"))
    if len(rb_tokens & tr_tokens) >= max(1, min(len(rb_tokens), 2)):
        return 1
    return 0


## 2. Read the raw CSV files

In [5]:
assign = read_csv_fallback(ASSIGN_PATH)
pre = read_csv_fallback(PRE_PATH)
post = read_csv_fallback(POST_PATH)

print("Assignment shape:", assign.shape)
print("Pre shape:", pre.shape)
print("Post shape:", post.shape)

assign.head(2)
pre.head(2)
post.head(2)


Assignment shape: (140, 24)
Pre shape: (132, 58)
Post shape: (132, 87)


,Timestamp,Student ID ( 学籍番号 ),0.1 Which ad-delivery method did you experience most clearly today? ( 今日、最もはっきりと体験した広告の提示方法はどれですか？),"0.2 For each ad below, did you notice it during the task? ( 以下の各広告について、実験中に気づいたものはありますか？) [Beverage Ad (飲料広告 )]","0.2 For each ad below, did you notice it during the task? ( 以下の各広告について、実験中に気づいたものはありますか？) [Chocolate Ad (チョコレート広告)]","0.2 For each ad below, did you notice it during the task? ( 以下の各広告について、実験中に気づいたものはありますか？) [Chips/cup noodles/snacks Ad (スナック(チップス、ラーメンカップ麺)広告)]",Confirmation,Mental Demand ( 知的 知覚的要求 ) (0 = Very Low 小さい–100 = Very High 大きい)\nHow mentally demanding was the task?\nどの程度、精神的に負担がありましたか?,Physical Demand ( 身体的要求 ) (0 = Very Low 小さい–100 = Very High 大きい)\nHow physically demanding was the task?\nどの程度、身体的に負担がありましたか?,Temporal Demand (タイムプレッシャー) (0 = Very Low 弱い–100 = Very High 強い)\nHow hurried or rushed was the pace of the task?\nペースはどの程度、急いでいましたか？,Performance ( 作業成績 ) (0 = Perfect 良い–100 = Failure 悪い)\nHow successful were you in accomplishing what you were asked to do?\n指示された内容をどの程度うまく達成できましたか？,Effort (努力) ( 0 = Very Low 少ない–100 = Very High 多い )\nHow hard did you have to work to accomplish your level of performance?\n達成のためにどの程度努力が必要でしたか？,"Frustration (フラストレーション) ( 0 = Very Low 低い–100 = Very High 高い )\nHow insecure, discouraged, irritated, stressed, and annoyed were you?\nどの程度、不安落胆いらだちストレスを感じましたか？",01. Mental Demand vs Physical Demand\n 知的 ⋅ 知覚的要求 vs 身体的要求,02. Mental Demand vs Temporal Demand\n 知的 ⋅ 知覚的要求 vs タイムプレッシャー\n,03. Mental Demand vs Performance\n 知的 ⋅ 知覚的要求 vs 作業成績,04. Mental Demand vs Effort\n 知的 ⋅ 知覚的要求 vs 努力,05. Mental Demand vs Frustration\n 知的 ⋅ 知覚的要求 vs フラストレーション,06. Physical Demand vs Temporal Demand\n 身体的要求 vs タイムプレッシャー,07. Physical Demand vs Performance\n 身体的要求 vs 作業成績,08. Physical Demand vs Effort\n 身体的要求 vs 努力,09. Physical Demand vs Frustration\n 身体的要求 vs フラストレーション,10. Temporal Demand vs Performance\n タイムプレッシャー vs 作業成績,11. Temporal Demand vs Effort\n タイムプレッシャー vs 努力,12. Temporal Demand vs Frustration\n タイムプレッシャー vs フラストレーション,13. Performance vs Effort\n 作業成績 vs 努力,14. Performance vs Frustration\n 作業成績 vs フラストレーション,15. Effort vs Frustration\n 努力 vs フラストレーション,"3.1 UES-SF (1=Strongly disagree, 5=Strongly agree)\nUES-SF(1=まったくそう思わない、5=非常にそう思う) [I lost myself in this experience. この体験に没頭した。]","3.1 UES-SF (1=Strongly disagree, 5=Strongly agree)\nUES-SF(1=まったくそう思わない、5=非常にそう思う) [The time I spent on this shopping task just slipped away. この買い物体験に費やした時間はあっという間に過ぎた。]","3.1 UES-SF (1=Strongly disagree, 5=Strongly agree)\nUES-SF(1=まったくそう思わない、5=非常にそう思う) [I was absorbed in this experience. この体験にのめり込んだ。]","3.1 UES-SF (1=Strongly disagree, 5=Strongly agree)\nUES-SF(1=まったくそう思わない、5=非常にそう思う) [I felt frustrated while doing this shopping task. この買い物体験中、イライラした。]","3.1 UES-SF (1=Strongly disagree, 5=Strongly agree)\nUES-SF(1=まったくそう思わない、5=非常にそう思う) [I found this shopping task confusing to do. この買い物体験は分かりにくかった。]","3.1 UES-SF (1=Strongly disagree, 5=Strongly agree)\nUES-SF(1=まったくそう思わない、5=非常にそう思う) [Doing this shopping task was taxing. この買い物体験は負担だった。]","3.1 UES-SF (1=Strongly disagree, 5=Strongly agree)\nUES-SF(1=まったくそう思わない、5=非常にそう思う) [This shopping task was attractive. この買い物体験は魅力的だった。]","3.1 UES-SF (1=Strongly disagree, 5=Strongly agree)\nUES-SF(1=まったくそう思わない、5=非常にそう思う) [This shopping task was aesthetically appealing. この買い物体験は視覚的に心地よかった。]","3.1 UES-SF (1=Strongly disagree, 5=Strongly agree)\nUES-SF(1=まったくそう思わない、5=非常にそう思う) [This shopping task appealed to my senses. この買い物体験は、感覚的にすごく良かった。]","3.1 UES-SF (1=Strongly disagree, 5=Strongly agree)\nUES-SF(1=まったくそう思わない、5=非常にそう思う) [Doing this shopping task was worthwhile. この買い物体験はやる価値があった。]","3.1 UES-SF (1=Strongly disagree, 5=Strongly agree)\nUES-SF(1=まったくそう思わない、5=非常にそう思う) [My experience was rewarding. 今回の体験は有意義だった。]","3.1 UES-SF (1=Strongly disagree, 5=Strongly agree)\nUES-SF(1=まったくそう思わない、5=非常にそう思う) [I felt interested in this experience. この体験に興味を持った。]","Overall, the advertisements I saw in this experiment were… \nこの実験で見た広告全体について... [Dislike … L

## 3. Normalize Student IDs and group labels

In [9]:
assign["id_norm"] = assign["Student ID"].apply(normalize_id)
pre["id_norm"] = pre["Student ID ( 学籍番号 )"].apply(normalize_id)
post["id_norm"] = post["Student ID ( 学籍番号 )"].apply(normalize_id)

assign["group"] = assign["Assigned Group"].apply(normalize_group)

assign = assign[assign["id_norm"].notna()].copy()
assign = assign[~assign["id_norm"].isin(["NAN", "TIMESTARTANDTIMEENDBASEDONARGLASSESVIDEOS"])].copy()
assign = assign.drop_duplicates(subset="id_norm", keep="first")

print("Assignment unique normalized IDs:", assign["id_norm"].nunique())
print("Pre unique normalized IDs:", pre["id_norm"].nunique())
print("Post unique normalized IDs:", post["id_norm"].nunique())

assign[["Student ID","id_norm","Assigned Group","group"]].head(10)


Assignment unique normalized IDs: 132
Pre unique normalized IDs: 132
Post unique normalized IDs: 132


,Student ID,id_norm,Assigned Group,group
0,C5IM2066,C5IM2066,Smartwatch,SMARTWATCH
1,Sato,SATO,Control,CONTROL
2,Wang,WANG,Control,CONTROL
3,Daiki,DAIKI,Smartwatch,SMARTWATCH
4,C4ID2014,C4ID2014,Smartwatch,SMARTWATCH
5,C5IM2502,C5IM2502,Control,CONTROL
6,C4IM2022,C4IM2022,AR,AR
7,C4TD9112,C4TD9112,AR,AR
8,C5IE2503,C5IE2503,Control,CONTROL
9,C4SM6701,C4SM6701,AR,AR


## 4. Merge assignment + pre + post

In [10]:
matched = assign.merge(pre, on="id_norm", how="inner", suffixes=("", "_pre"))
matched = matched.merge(post, on="id_norm", how="inner", suffixes=("", "_post"))
matched["student_id"] = matched["id_norm"]

print("Matched subject count:", matched["student_id"].nunique())
matched["group"].value_counts()


Matched subject count: 132


group
SMARTWATCH    44
CONTROL       44
AR            44
Name: count, dtype: int64

In [ ]:
group_id_df = (
    matched[["group", "student_id"]]
    .drop_duplicates()
    .sort_values(["group", "student_id"])
    .reset_index(drop=True)
)

# group_id_df.to_csv("student_ids_by_group.csv", index=False)

group_id_df

,group,student_id
0,AR,-
1,AR,12345
2,AR,ABCDEFG
3,AR,C2TB5066
4,AR,C2TB5069
...,...,...
127,SMARTWATCH,C5TM9603
128,SMARTWATCH,C6IM3011
129,SMARTWATCH,DAIKI
130,SMARTWATCH,HONGTHONGPONTAKORN


## 5. Build pre-questionnaire composites

In [ ]:
# Impulse buying
impulse_cols = find_cols(pre, ["Impulse buying tendency"], exclude=["Timestamp","Student ID"])
impulse_cols = [c for c in matched.columns if c in impulse_cols]
matched["impulse_buying"] = safe_mean(matched, impulse_cols)

# PoP attitude
pop_cols = [c for c in matched.columns if "In-store advertising attitude" in c]
matched["pop_attitude"] = safe_mean(matched, pop_cols)

# TAEG-S with reverse coding
taeg_cols = [c for c in matched.columns if "Below you will find a series of statements" in c]
taeg_neg_markers = ["make people sick", "intellectual impoverishment", "cause stress"]
taeg_use_cols = []
for c in taeg_cols:
    matched[c] = pd.to_numeric(matched[c], errors="coerce")
    if any(m in c.lower() for m in taeg_neg_markers):
        newc = c + "__rev"
        matched[newc] = reverse_code(matched[c], 1, 5)
        taeg_use_cols.append(newc)
    else:
        taeg_use_cols.append(c)

matched["taeg"] = safe_mean(matched, taeg_use_cols)

matched[["student_id","group","impulse_buying","pop_attitude","taeg"]].head()


## 6. Build post-questionnaire composites

In [ ]:
# Manipulation check
manip_col = find_cols(post, ["Which ad-delivery method did you experience most clearly"])[0]
matched["manip_group"] = matched[manip_col].apply(parse_manipulation_group)

# Noticing
bev_notice_col = [c for c in matched.columns if "notice it during the task" in c and "Beverage Ad" in c][0]
choc_notice_col = [c for c in matched.columns if "notice it during the task" in c and "Chocolate Ad" in c][0]
snack_notice_col = [c for c in matched.columns if "notice it during the task" in c and "Chips/cup noodles/snacks" in c][0]

matched["noticed_beverage"] = matched[bev_notice_col].apply(parse_yes_no_not_sure)
matched["noticed_chocolate"] = matched[choc_notice_col].apply(parse_yes_no_not_sure)
matched["noticed_snack"] = matched[snack_notice_col].apply(parse_yes_no_not_sure)
matched["noticed_any"] = matched[["noticed_beverage","noticed_chocolate","noticed_snack"]].max(axis=1)
matched["noticed_n"] = matched[["noticed_beverage","noticed_chocolate","noticed_snack"]].fillna(0).sum(axis=1)
matched["noticed_all3"] = (matched["noticed_n"] == 3).astype(int)

# NASA-TLX
rating_cols = {
    "mental": [c for c in matched.columns if c.startswith("Mental Demand")][0],
    "physical": [c for c in matched.columns if c.startswith("Physical Demand")][0],
    "temporal": [c for c in matched.columns if c.startswith("Temporal Demand")][0],
    "performance": [c for c in matched.columns if c.startswith("Performance")][0],
    "effort": [c for c in matched.columns if c.startswith("Effort")][0],
    "frustration": [c for c in matched.columns if c.startswith("Frustration")][0],
}
pair_cols = [c for c in matched.columns if re.match(r"^\d{2}\.", str(c))]
matched["tlx_weighted"] = weighted_tlx(matched, rating_cols, pair_cols)

# UES-SF
ues_cols = [c for c in matched.columns if "UES-SF" in c]
ues_neg_markers = ["frustrated", "confusing", "taxing"]
ues_use_cols = []
for c in ues_cols:
    matched[c] = pd.to_numeric(matched[c], errors="coerce")
    if any(m in c.lower() for m in ues_neg_markers):
        newc = c + "__rev"
        matched[newc] = reverse_code(matched[c], 1, 5)
        ues_use_cols.append(newc)
    else:
        ues_use_cols.append(c)
matched["ues"] = safe_mean(matched, ues_use_cols)

# Aad
aad_cols = [c for c in matched.columns if "Overall, the advertisements I saw in this experiment were" in c]
for c in aad_cols:
    matched[c] = pd.to_numeric(matched[c], errors="coerce")
matched["aad"] = safe_mean(matched, aad_cols)

matched[["student_id","group","tlx_weighted","ues","aad","noticed_n"]].head()


## 7. Reshape the repeated ad blocks into long format

In [ ]:
true_ad_cols = {1:"True AD1", 2:"True AD2", 3:"True AD3"}
true_buy_cols = {1:"True Buy AD1 Item after exposure", 2:"True Buy AD2 Item after exposure", 3:"True Buy AD3 Item after exposure"}

rows = []
for ad_i in [1,2,3]:
    prefix = f"[AD {ad_i}"
    ad_cols = [c for c in matched.columns if c.startswith(prefix)]

    pi_cols = [c for c in ad_cols if "Purchase intention" in c]
    inc_cols = [c for c in ad_cols if "Perceived incentive / discount influence" in c]
    exp_col = [c for c in ad_cols if "Exposure check" in c][0]
    cat_col = [c for c in ad_cols if "What product category was advertised" in c][0]
    brand_col = [c for c in ad_cols if "What brand was shown" in c][0]
    disc_col = [c for c in ad_cols if "What discount was offered" in c][0]

    tmp = matched[[
        "student_id","group","pop_attitude","taeg","ues","aad","tlx_weighted",
        true_ad_cols[ad_i], true_buy_cols[ad_i]
    ] + pi_cols + inc_cols + [exp_col, cat_col, brand_col, disc_col]].copy()

    tmp["ad_index"] = ad_i
    tmp["true_ad"] = matched[true_ad_cols[ad_i]]
    tmp["true_discount"] = tmp["true_ad"].apply(parse_discount_from_ad)
    tmp["true_root"] = tmp["true_ad"].apply(product_root)
    tmp["true_category"] = tmp["true_root"].apply(product_category_from_root)

    tmp["bought_after_exposure"] = tmp[true_buy_cols[ad_i]].astype(str).str.strip().str.lower().map({"yes":1, "no":0})

    for c in pi_cols + inc_cols:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
    tmp["purchase_intention"] = tmp[pi_cols].mean(axis=1)
    tmp["perceived_incentive"] = tmp[inc_cols].mean(axis=1)
    tmp["ad_exposure_check"] = tmp[exp_col].apply(parse_yes_no_not_sure)

    tmp["resp_category"] = tmp[cat_col].apply(normalize_category_response)
    tmp["resp_brand"] = tmp[brand_col]
    tmp["resp_discount"] = tmp[disc_col].apply(normalize_discount_response)

    tmp["mem_category"] = np.where(tmp["resp_category"].notna() & (tmp["resp_category"] == tmp["true_category"]), 1, 0)
    tmp["mem_brand"] = [brand_match(rb, tr) for rb, tr in zip(tmp["resp_brand"], tmp["true_root"])]
    tmp["mem_discount"] = np.where(tmp["resp_discount"].notna() & (tmp["resp_discount"] == tmp["true_discount"]), 1, 0)
    tmp["ad_memory"] = pd.DataFrame({
        "mem_category": tmp["mem_category"],
        "mem_brand": tmp["mem_brand"],
        "mem_discount": tmp["mem_discount"]
    }).sum(axis=1, skipna=True)

    rows.append(tmp[[
        "student_id","group","ad_index","true_discount","true_category","true_root",
        "purchase_intention","perceived_incentive","ad_exposure_check","ad_memory",
        "bought_after_exposure","pop_attitude","taeg","ues","aad","tlx_weighted"
    ]])

ad_long = pd.concat(rows, ignore_index=True)
print("Ad-level rows:", len(ad_long))
ad_long.head()


## 8. Build subject-level outcomes used in the no-gaze tests

In [ ]:
mem_subject = ad_long.groupby("student_id", as_index=False)["ad_memory"].mean().rename(columns={"ad_memory":"ad_memory_mean"})
pi_subject = ad_long.groupby("student_id", as_index=False)["purchase_intention"].mean().rename(columns={"purchase_intention":"purchase_intention_overall"})
inc_subject = ad_long.groupby("student_id", as_index=False)["perceived_incentive"].mean().rename(columns={"perceived_incentive":"perceived_incentive_overall"})
buy_subject = ad_long.groupby("student_id", as_index=False)["bought_after_exposure"].max().rename(columns={"bought_after_exposure":"purchased_any_advertised"})

matched2 = matched.merge(mem_subject, on="student_id", how="left")
matched2 = matched.merge(pi_subject, on="student_id", how="left")
matched2 = matched2.merge(inc_subject, on="student_id", how="left")
matched2 = matched2.merge(buy_subject, on="student_id", how="left")
matched2["basket_size"] = pd.to_numeric(matched2["Total Item Number"], errors="coerce")

matched2[["student_id","group","ad_memory_mean","purchase_intention_overall","purchased_any_advertised","basket_size"]].head()


## 9. Reliability analysis

In [ ]:
reliability_rows = []

reliability_rows.append({
    "scale":"impulse_buying",
    "n_items": len(impulse_cols),
    "alpha": cronbach_alpha(to_numeric_df(matched, impulse_cols))
})
reliability_rows.append({
    "scale":"pop_attitude",
    "n_items": len(pop_cols),
    "alpha": cronbach_alpha(to_numeric_df(matched, pop_cols))
})
reliability_rows.append({
    "scale":"taeg",
    "n_items": len(taeg_use_cols),
    "alpha": cronbach_alpha(to_numeric_df(matched, taeg_use_cols))
})
reliability_rows.append({
    "scale":"ues",
    "n_items": len(ues_use_cols),
    "alpha": cronbach_alpha(to_numeric_df(matched, ues_use_cols))
})
reliability_rows.append({
    "scale":"aad",
    "n_items": len(aad_cols),
    "alpha": cronbach_alpha(to_numeric_df(matched, aad_cols))
})

pi_item_rows = []
inc_item_rows = []
for ad_i in [1,2,3]:
    prefix = f"[AD {ad_i}"
    ad_cols = [c for c in matched.columns if c.startswith(prefix)]
    pi_cols = [c for c in ad_cols if "Purchase intention" in c]
    inc_cols = [c for c in ad_cols if "Perceived incentive / discount influence" in c]
    pi_tmp = to_numeric_df(matched, pi_cols)
    inc_tmp = to_numeric_df(matched, inc_cols)
    pi_tmp.columns = [f"pi_{j+1}" for j in range(pi_tmp.shape[1])]
    inc_tmp.columns = [f"inc_{j+1}" for j in range(inc_tmp.shape[1])]
    pi_item_rows.append(pi_tmp)
    inc_item_rows.append(inc_tmp)

pi_all = pd.concat(pi_item_rows, ignore_index=True)
inc_all = pd.concat(inc_item_rows, ignore_index=True)

reliability_rows.append({
    "scale":"purchase_intention_pooled",
    "n_items": pi_all.shape[1],
    "alpha": cronbach_alpha(pi_all)
})
reliability_rows.append({
    "scale":"perceived_incentive_pooled",
    "n_items": inc_all.shape[1],
    "alpha": cronbach_alpha(inc_all)
})

reliability_df = pd.DataFrame(reliability_rows)
reliability_df


## 10. Manipulation checks

In [ ]:
manip_ct = pd.crosstab(matched["group"], matched["manip_group"], dropna=False)
manip_ct


In [ ]:
chi2, p, dof, expected = chi2_contingency(manip_ct)
print("Chi-square p-value:", p)

manip_summary = []
for g in ["CONTROL","SMARTWATCH","AR"]:
    sub = matched.loc[matched["group"] == g]
    correct = (sub["group"] == sub["manip_group"]).sum()
    manip_summary.append({
        "group": g,
        "n": len(sub),
        "n_correct": int(correct),
        "pct_correct": 100.0 * correct / len(sub)
    })
manip_summary_df = pd.DataFrame(manip_summary)
manip_summary_df


### Figure: Manipulation confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
im = ax.imshow(manip_ct.values)

ax.set_xticks(range(manip_ct.shape[1]), labels=manip_ct.columns, rotation=30, ha="right")
ax.set_yticks(range(manip_ct.shape[0]), labels=manip_ct.index)
ax.set_xlabel("Manipulation check response")
ax.set_ylabel("Assigned group")
ax.set_title("Manipulation Check Confusion Matrix")

for i in range(manip_ct.shape[0]):
    for j in range(manip_ct.shape[1]):
        ax.text(j, i, manip_ct.values[i, j], ha="center", va="center")

fig.tight_layout()
plt.show()


### Ad noticing by group

In [ ]:
notice_rows = []
for var in ["noticed_beverage","noticed_chocolate","noticed_snack","noticed_any","noticed_all3"]:
    ct = pd.crosstab(matched["group"], matched[var])
    p = chi2_contingency(ct)[1] if ct.shape[1] >= 2 else np.nan
    for g in ["CONTROL","SMARTWATCH","AR"]:
        sub = matched.loc[matched["group"] == g]
        notice_rows.append({
            "measure": var,
            "group": g,
            "n": len(sub),
            "n_yes": int((sub[var] == 1).sum()),
            "pct_yes": 100.0 * (sub[var] == 1).mean(),
            "chi2_p_overall": p
        })
notice_df = pd.DataFrame(notice_rows)
notice_df


### Figure: percent noticing by group

In [ ]:
plot_df = notice_df[notice_df["measure"].isin(["noticed_beverage","noticed_chocolate","noticed_snack","noticed_any","noticed_all3"])].copy()

measures = ["noticed_beverage","noticed_chocolate","noticed_snack","noticed_any","noticed_all3"]
groups = ["CONTROL","SMARTWATCH","AR"]
x = np.arange(len(measures))
width = 0.25

fig, ax = plt.subplots(figsize=(9,5))

for idx, g in enumerate(groups):
    y = [plot_df[(plot_df["measure"] == m) & (plot_df["group"] == g)]["pct_yes"].iloc[0] for m in measures]
    ax.bar(x + (idx-1)*width, y, width=width, label=g)

ax.set_xticks(x, ["Beverage", "Chocolate", "Snack", "Any ad", "All 3 ads"])
ax.set_ylabel("Percent noticed")
ax.set_title("Ad Noticing by Group")
ax.legend()
fig.tight_layout()
plt.show()


## 11. Subject-level descriptives

In [ ]:
subject_descriptives = matched2.groupby("group").agg(
    n=("student_id","nunique"),
    ad_memory_mean=("ad_memory_mean","mean"),
    purchase_intention_overall=("purchase_intention_overall","mean"),
    purchased_any_advertised=("purchased_any_advertised","mean"),
    basket_size=("basket_size","mean"),
    ues=("ues","mean"),
    aad=("aad","mean"),
    pop_attitude=("pop_attitude","mean"),
    taeg=("taeg","mean"),
    tlx_weighted=("tlx_weighted","mean"),
    noticed_any=("noticed_any","mean"),
    noticed_n=("noticed_n","mean")
).reset_index()

subject_descriptives


### Figure: selected subject-level means by group

In [ ]:
plot_vars = ["ad_memory_mean","purchase_intention_overall","purchased_any_advertised","basket_size","ues","aad","tlx_weighted"]
n_vars = len(plot_vars)

fig = plt.figure(figsize=(12, 10))
for i, v in enumerate(plot_vars, start=1):
    ax = fig.add_subplot((n_vars + 1)//2, 2, i)
    ax.bar(subject_descriptives["group"], subject_descriptives[v])
    ax.set_title(v)
plt.tight_layout()
plt.show()


## 12. Ad-level descriptives

In [ ]:
ad_descriptives = ad_long.groupby(["group","true_discount"]).agg(
    n=("student_id","size"),
    purchase_intention=("purchase_intention","mean"),
    perceived_incentive=("perceived_incentive","mean"),
    ad_memory=("ad_memory","mean"),
    bought_after_exposure=("bought_after_exposure","mean"),
    ad_exposure_check=("ad_exposure_check","mean")
).reset_index()

ad_descriptives


### Figure: purchase intention by discount

In [ ]:
discount_means = ad_long.groupby("true_discount", as_index=False)["purchase_intention"].mean()

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(discount_means["true_discount"], discount_means["purchase_intention"])
ax.set_xlabel("Discount type")
ax.set_ylabel("Mean purchase intention")
ax.set_title("Ad-level Purchase Intention by Discount")
fig.tight_layout()
plt.show()


### Figure: perceived incentive by discount

In [ ]:
discount_inc = ad_long.groupby("true_discount", as_index=False)["perceived_incentive"].mean()

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(discount_inc["true_discount"], discount_inc["perceived_incentive"])
ax.set_xlabel("Discount type")
ax.set_ylabel("Mean perceived incentive")
ax.set_title("Perceived Incentive by Discount")
fig.tight_layout()
plt.show()


## 13. No-gaze hypothesis tests

In [ ]:
results = []

# H2
d = matched2[["group","ad_memory_mean"]].dropna()
m = robust_ols("ad_memory_mean ~ C(group)", d)
results.append({"hypothesis":"H2","question":"Does device condition affect ad memory?","tool":"OLS (HC3 robust SE)","N":len(d),"p_value":omnibus_p_for_factor(m,"C(group)")})

# H3a
d = matched2[["group","purchase_intention_overall"]].dropna()
m = robust_ols("purchase_intention_overall ~ C(group)", d)
results.append({"hypothesis":"H3a","question":"Does device condition affect overall purchase intention?","tool":"OLS (HC3 robust SE)","N":len(d),"p_value":omnibus_p_for_factor(m,"C(group)")})

# H3a1
d = ad_long[["student_id","group","purchase_intention"]].dropna()
m = clustered_ols("purchase_intention ~ C(group)", d, "student_id")
results.append({"hypothesis":"H3a1","question":"Does device condition affect ad-level purchase intention?","tool":"OLS with clustered SE by participant","N":len(d),"p_value":omnibus_p_for_factor(m,"C(group)")})

# H3b
d = matched2[["group","purchased_any_advertised"]].dropna()
m = robust_logit("purchased_any_advertised ~ C(group)", d)
results.append({"hypothesis":"H3b","question":"Does device condition affect purchase probability of an advertised item?","tool":"Logistic regression (HC3 robust SE)","N":len(d),"p_value":omnibus_p_for_factor(m,"C(group)")})

# H3b1
d = matched2[["group","basket_size"]].dropna()
m = robust_ols("basket_size ~ C(group)", d)
results.append({"hypothesis":"H3b1","question":"Does device condition affect basket size?","tool":"OLS (HC3 robust SE)","N":len(d),"p_value":omnibus_p_for_factor(m,"C(group)")})

# H4
d = ad_long[["student_id","true_discount","purchase_intention"]].dropna()
m = clustered_ols("purchase_intention ~ C(true_discount)", d, "student_id")
results.append({"hypothesis":"H4","question":"Does discount type affect purchase intention?","tool":"OLS with clustered SE by participant","N":len(d),"p_value":omnibus_p_for_factor(m,"C(true_discount)")})

# H5
d = ad_long[["student_id","group","true_discount","purchase_intention"]].dropna()
m = clustered_ols("purchase_intention ~ C(group) * C(true_discount)", d, "student_id")
interaction_names = [n for n in m.params.index if "C(group)" in n and "C(true_discount)" in n]
if interaction_names:
    R = np.zeros((len(interaction_names), len(m.params)))
    for i, name in enumerate(interaction_names):
        R[i, list(m.params.index).index(name)] = 1
    p_interaction = float(m.wald_test(R).pvalue)
else:
    p_interaction = np.nan
results.append({"hypothesis":"H5","question":"Does the effect of discount framing depend on device condition?","tool":"OLS with clustered SE by participant","N":len(d),"p_value":p_interaction})

# H6
d = matched2[["group","ues"]].dropna()
m = robust_ols("ues ~ C(group)", d)
results.append({"hypothesis":"H6","question":"Does device condition affect engagement (UES)?","tool":"OLS (HC3 robust SE)","N":len(d),"p_value":omnibus_p_for_factor(m,"C(group)")})

# H7
d = matched2[["group","aad"]].dropna()
m = robust_ols("aad ~ C(group)", d)
results.append({"hypothesis":"H7","question":"Does device condition affect attitude toward the ad (Aad)?","tool":"OLS (HC3 robust SE)","N":len(d),"p_value":omnibus_p_for_factor(m,"C(group)")})

# H8
d = ad_long[["student_id","true_discount","perceived_incentive"]].dropna()
m = clustered_ols("perceived_incentive ~ C(true_discount)", d, "student_id")
results.append({"hypothesis":"H8","question":"Does discount type affect perceived incentive?","tool":"OLS with clustered SE by participant","N":len(d),"p_value":omnibus_p_for_factor(m,"C(true_discount)")})

# H9
d = matched2[["group","taeg","purchase_intention_overall"]].dropna()
m = robust_ols("purchase_intention_overall ~ C(group) * taeg", d)
interaction_names = [n for n in m.params.index if "C(group)" in n and ":taeg" in n]
if interaction_names:
    R = np.zeros((len(interaction_names), len(m.params)))
    for i, name in enumerate(interaction_names):
        R[i, list(m.params.index).index(name)] = 1
    p_h9 = float(m.wald_test(R).pvalue)
else:
    p_h9 = np.nan
results.append({"hypothesis":"H9","question":"Does tech affinity moderate device effects on purchase intention?","tool":"OLS (HC3 robust SE)","N":len(d),"p_value":p_h9})

# H10a
d = matched2[["group","pop_attitude","purchase_intention_overall"]].dropna()
m = robust_ols("purchase_intention_overall ~ C(group) * pop_attitude", d)
interaction_names = [n for n in m.params.index if "C(group)" in n and ":pop_attitude" in n]
if interaction_names:
    R = np.zeros((len(interaction_names), len(m.params)))
    for i, name in enumerate(interaction_names):
        R[i, list(m.params.index).index(name)] = 1
    p_h10a = float(m.wald_test(R).pvalue)
else:
    p_h10a = np.nan
results.append({"hypothesis":"H10a","question":"Does baseline PoP attitude moderate device effects on purchase intention?","tool":"OLS (HC3 robust SE)","N":len(d),"p_value":p_h10a})

# H10b
d = matched2[["group","pop_attitude","aad"]].dropna()
m = robust_ols("aad ~ C(group) * pop_attitude", d)
interaction_names = [n for n in m.params.index if "C(group)" in n and ":pop_attitude" in n]
if interaction_names:
    R = np.zeros((len(interaction_names), len(m.params)))
    for i, name in enumerate(interaction_names):
        R[i, list(m.params.index).index(name)] = 1
    p_h10b = float(m.wald_test(R).pvalue)
else:
    p_h10b = np.nan
results.append({"hypothesis":"H10b","question":"Does baseline PoP attitude moderate device effects on Aad?","tool":"OLS (HC3 robust SE)","N":len(d),"p_value":p_h10b})

# H11a
d = matched2[["tlx_weighted","ues"]].dropna()
m = robust_ols("ues ~ tlx_weighted", d)
results.append({"hypothesis":"H11a","question":"Is higher workload associated with lower engagement?","tool":"OLS (HC3 robust SE)","N":len(d),"p_value":float(m.pvalues.get("tlx_weighted", np.nan))})

# H11b
d = matched2[["tlx_weighted","aad"]].dropna()
m = robust_ols("aad ~ tlx_weighted", d)
results.append({"hypothesis":"H11b","question":"Is higher workload associated with worse Aad?","tool":"OLS (HC3 robust SE)","N":len(d),"p_value":float(m.pvalues.get("tlx_weighted", np.nan))})

# H12a
d = matched2[["pop_attitude","purchase_intention_overall"]].dropna()
m = robust_ols("purchase_intention_overall ~ pop_attitude", d)
results.append({"hypothesis":"H12a","question":"Does baseline PoP attitude positively predict purchase intention?","tool":"OLS (HC3 robust SE)","N":len(d),"p_value":float(m.pvalues.get("pop_attitude", np.nan))})

# H12b
d = matched2[["pop_attitude","aad"]].dropna()
m = robust_ols("aad ~ pop_attitude", d)
results.append({"hypothesis":"H12b","question":"Does baseline PoP attitude positively predict Aad?","tool":"OLS (HC3 robust SE)","N":len(d),"p_value":float(m.pvalues.get("pop_attitude", np.nan))})

results_df = pd.DataFrame(results)
results_df["significant_0_05"] = results_df["p_value"] < 0.05
results_df


### Figure: p-values for no-gaze hypothesis tests

In [ ]:
plot_df = results_df.copy().sort_values("p_value")
fig, ax = plt.subplots(figsize=(10,6))
ax.barh(plot_df["hypothesis"], plot_df["p_value"])
ax.axvline(0.05, linestyle="--")
ax.set_xlabel("p-value")
ax.set_ylabel("Hypothesis")
ax.set_title("No-gaze Hypothesis Test p-values")
fig.tight_layout()
plt.show()


## 14. Save outputs to CSV

In [ ]:
reliability_df.to_csv(OUTPUT_DIR / "reliability_summary.csv", index=False)
manip_ct.to_csv(OUTPUT_DIR / "manipulation_confusion_matrix.csv")
manip_summary_df.to_csv(OUTPUT_DIR / "manipulation_summary.csv", index=False)
notice_df.to_csv(OUTPUT_DIR / "noticing_by_group.csv", index=False)
subject_descriptives.to_csv(OUTPUT_DIR / "subject_level_descriptives_by_group.csv", index=False)
ad_descriptives.to_csv(OUTPUT_DIR / "ad_level_descriptives_by_group_discount.csv", index=False)
results_df.to_csv(OUTPUT_DIR / "hypothesis_results_no_gaze.csv", index=False)

print("Saved files to:", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", p.name)


## 15. Optional: save figures to PNG

In [ ]:
# Example for saving current / repeated figures:
# fig.savefig(OUTPUT_DIR / "figure_name.png", dpi=200, bbox_inches="tight")
print("Uncomment fig.savefig(...) lines in the plotting cells if you want PNG outputs.")
